# Reliability-Aware Hybrid Quantum Routing Optimization in Heterogeneous Quantum Networks (Top-k Routing)

This project investigates routing optimization in heterogeneous quantum communication networks using both classical and quantum algorithms. Unlike shortest-path routing, the framework explores multiple candidate paths between source and destination nodes through Top-k path selection. This enables routing decisions to be made from a larger search space, improving the ability to identify globally reliable communication paths under practical quantum communication constraints.

The framework incorporates quantum memory lifetime (τ), entanglement generation rate (λ), and routing reliability into the optimization process. Classical approaches including Greedy, DP, MRAG, and MRADP are extended to operate on Top-k candidate paths and are compared with QAOA-based quantum optimization. The QAOA model is implemented using Qiskit and validated on both Aer Simulator and real IBM Quantum hardware.

# Library Installation and Environment Setup

In [ ]:
# ============================================================
# Install Required Libraries
# ============================================================

# Qiskit          → Quantum circuit design and simulation
# Qiskit Aer      → Quantum simulator backend
# NetworkX        → Quantum network graph generation
# Matplotlib      → Graphs and visualization
# NumPy           → Numerical computations
# Pandas          → Data handling and analysis
# pylatexenc      → Proper quantum circuit rendering

!pip install qiskit
!pip install qiskit-aer
!pip install networkx
!pip install matplotlib
!pip install numpy
!pip install pandas
!pip install pylatexenc

In [ ]:
# ============================================================
# Import Required Libraries
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator


# Quantum Reliability Modeling using Qiskit

In [ ]:
# ============================================================
# RT_u using Quantum Teleportation with Noise Modeling
# ============================================================

from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    thermal_relaxation_error
)

import numpy as np


def compute_RT(tau, lam,
               p_error=0.01,
               shots=1024,
               show_circuit=True):

    """
    Compute teleportation reliability RT_u
    """

    # ── Quantum Teleportation Circuit ─────────────────────

    qc = QuantumCircuit(3, 3)

    qc.x(0)              # Input state |1⟩

    # Bell pair generation
    qc.h(1)
    qc.cx(1, 2)

    # Bell-state measurement
    qc.cx(0, 1)
    qc.h(0)

    qc.measure(0, 0)
    qc.measure(1, 1)

    # Correction operations
    qc.cx(1, 2)
    qc.cz(0, 2)

    # Idle gate for decoherence modeling
    qc.id(2)

    qc.measure(2, 2)

    # ── Display Circuit ───────────────────────────────────

    if show_circuit:

        print("\nQuantum Teleportation Circuit:\n")

        try:
            display(qc.draw("mpl"))

        except:
            print(qc.draw())

    # ── Noise Model ───────────────────────────────────────

    noise_model = NoiseModel()

    # Gate depolarization noise
    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(p_error, 1),
        ['h', 'x']
    )

    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(p_error, 2),
        ['cx']
    )

    # Memory decoherence noise
    T1 = tau
    T2 = tau / 2

    idle_time = 1.0 / lam

    thermal_error = thermal_relaxation_error(
        T1,
        T2,
        idle_time
    )

    noise_model.add_all_qubit_quantum_error(
        thermal_error,
        ['id']
    )

    # ── Run Simulation ────────────────────────────────────

    sim = AerSimulator(
        noise_model=noise_model,
        seed_simulator=42
    )

    result = sim.run(
        qc,
        shots=shots
    ).result()

    counts = result.get_counts()

    # ── Compute Teleportation Reliability ────────────────

    success = 0

    for bits, count in counts.items():

        # c2 is output qubit
        if bits[0] == '1':
            success += count

    RT = success / shots

    return RT

In [ ]:
# ============================================================
# Example Execution of RT_u Computation
# ============================================================

# tau → Quantum memory lifetime
# lam → Entanglement generation rate

tau = 8.0
lam = 15.0

# Compute teleportation reliability
RT_u = compute_RT(tau, lam)

print(f"Teleportation Reliability (RT_u) = {RT_u:.4f}")

In [ ]:
# ============================================================
# RH_u using 3-Qubit Redundancy with Majority Voting
# ============================================================

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    thermal_relaxation_error
)

import numpy as np


def compute_RH(tau, lam,
               p_error=0.01,
               shots=1024,
               show_circuit=True):

    """
    Compute hop-by-hop reliability RH_u
    using repetition coding and majority voting
    """

    # ── 3-Qubit Redundancy Circuit ───────────────────────

    qc = QuantumCircuit(3, 3)

    qc.x(0)

    # Encode redundant copies
    qc.cx(0, 1)
    qc.cx(0, 2)

    # Idle gates for decoherence modeling
    qc.id(0)
    qc.id(1)
    qc.id(2)

    # Measurements
    qc.measure(0, 0)
    qc.measure(1, 1)
    qc.measure(2, 2)

    # ── Display Circuit ───────────────────────────────────

    if show_circuit:

        print("\n3-Qubit Redundancy Circuit:\n")

        try:
            display(qc.draw("mpl"))

        except:
            print(qc.draw())

    # ── Noise Model ───────────────────────────────────────

    noise_model = NoiseModel()

    # Gate depolarization noise
    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(p_error, 1),
        ['x']
    )

    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(p_error, 2),
        ['cx']
    )

    # Memory decoherence noise
    T1 = tau
    T2 = tau / 2

    idle_time = 1.0 / lam

    thermal_error = thermal_relaxation_error(
        T1,
        T2,
        idle_time
    )

    noise_model.add_all_qubit_quantum_error(
        thermal_error,
        ['id']
    )

    # ── Run Simulation ────────────────────────────────────

    sim = AerSimulator(
        noise_model=noise_model,
        seed_simulator=42
    )

    result = sim.run(
        qc,
        shots=shots
    ).result()

    counts = result.get_counts()

    # ── Majority Voting ──────────────────────────────────

    success = 0

    for bits, count in counts.items():

        # Majority voting condition
        ones = bits.count('1')

        if ones >= 2:
            success += count

    RH = success / shots

    return RH

In [ ]:
# ============================================================
# Example Execution of RH_u Computation
# ============================================================

# tau → Quantum memory lifetime
# lam → Entanglement generation rate

tau = 8.0
lam = 15.0

# Compute hop-by-hop reliability
RH_u = compute_RH(tau, lam)

print(f"Hop-by-Hop Reliability (RH_u) = {RH_u:.4f}")

In [ ]:
# ============================================================
# Cache for Quantum-Computed RT_u and RH_u
# ============================================================

# Cache avoids repeated quantum circuit simulation
# for same (tau, lambda) values

RT_CACHE = {}
RH_CACHE = {}


def _cache_key(tau, lam):

    # Stable rounding to avoid float precision mismatch
    return (
        round(float(tau), 3),
        round(float(lam), 3)
    )


def get_RT(tau, lam):

    key = _cache_key(tau, lam)

    # Compute only if value not already stored
    if key not in RT_CACHE:

        RT_CACHE[key] = compute_RT(
            tau,
            lam,
            show_circuit=False
        )

    return RT_CACHE[key]


def get_RH(tau, lam):

    key = _cache_key(tau, lam)

    # Compute only if value not already stored
    if key not in RH_CACHE:

        RH_CACHE[key] = compute_RH(
            tau,
            lam,
            show_circuit=False
        )

    return RH_CACHE[key]

# Quantum Network Creation and Node Parameter Assignment

In [ ]:
# ============================================================
# Quantum Hardware Platform Definitions
# ============================================================

# Fixed random seed for reproducibility
np.random.seed(42)

# ── Hardware Platform Definitions ────────────────────────

# Each hardware platform has:
# tau_range    → Quantum memory lifetime range
# lambda_range → Entanglement generation rate range

# Hardware configurations inspired by practical
# characteristics of superconducting, NV-center,
# and trapped-ion quantum platforms.
HARDWARE_TYPES = {

    'superconducting': {

        'tau_range': (5.0, 10.0),

        'lambda_range': (20.0, 30.0),

        'description':
        'Fast entanglement generation, short memory lifetime'
    },

    'nv_center': {

        'tau_range': (8.0, 15.0),

        'lambda_range': (12.0, 20.0),

        'description':
        'Balanced memory and entanglement performance'
    },

    'trapped_ion': {

        'tau_range': (20.0, 40.0),

        'lambda_range': (5.0, 10.0),

        'description':
        'Long memory lifetime, slower entanglement generation'
    }
}

# ── Hardware Occurrence Distribution ─────────────────────

# Probability distribution used to generate
# heterogeneous quantum network environments
HARDWARE_DISTRIBUTION = {

    'superconducting': 0.40,

    'nv_center': 0.35,

    'trapped_ion': 0.25
}


# ── Random Hardware Assignment ───────────────────────────

def assign_hardware_type():

    rand = np.random.random()

    cumulative = 0.0

    for hw_type, prob in HARDWARE_DISTRIBUTION.items():

        cumulative += prob

        if rand < cumulative:
            return hw_type

    return 'nv_center'

In [ ]:
# ============================================================
# Assign Quantum-Aware Parameters to Network Nodes
# ============================================================

def assign_node_parameters_quantum(G, bifunctional_prob=0.85):

    for node in G.nodes():

        # ── Node Type Assignment ──────────────────────────

        is_bi = np.random.rand() < bifunctional_prob

        G.nodes[node]['type'] = (
            'bi' if is_bi else 'mono'
        )

        # ── Hardware Assignment ───────────────────────────

        hw_type = assign_hardware_type()

        G.nodes[node]['hardware'] = hw_type

        hw = HARDWARE_TYPES[hw_type]

        # ── Quantum Memory & Entanglement Parameters ─────

        tau = np.random.uniform(*hw['tau_range'])
        lam = np.random.uniform(*hw['lambda_range'])

        G.nodes[node]['tau'] = tau
        G.nodes[node]['lambda'] = lam

        # Entanglement waiting time
        G.nodes[node]['W'] = 1.0 / lam

        # ── Quantum-derived Reliability Values ───────────

        RT_base = get_RT(tau, lam)
        RH_base = get_RH(tau, lam)

        # ── Controlled Competition Between Modes ─────────

        delta = np.random.uniform(0.01, 0.03)

        if np.random.rand() < 0.5:

            # Teleportation-favored node
            RT = min(0.995, RT_base + delta)
            RH = max(0.94, RH_base - delta/2)

        else:

            # H2H-favored node
            RH = min(0.995, RH_base + delta)
            RT = max(0.94, RT_base - delta/2)

        G.nodes[node]['RT'] = RT
        G.nodes[node]['RH'] = RH

        # ── Processing Delays ────────────────────────────

        G.nodes[node]['PT'] = np.random.uniform(0.3, 1.0)

        # H2H includes additional processing overhead
        G.nodes[node]['PH'] = (
            np.random.uniform(0.2, 0.9)
            + np.random.uniform(0.05, 0.15)
        )

    # ── Global Reliability Balancing ─────────────────────

    avg_RT = np.mean([
        G.nodes[n]['RT']
        for n in G.nodes()
    ])

    avg_RH = np.mean([
        G.nodes[n]['RH']
        for n in G.nodes()
    ])

    # Prevent one routing mode from dominating globally
    if abs(avg_RT - avg_RH) > 0.005:

        for n in G.nodes():

            G.nodes[n]['RT'] *= 0.995
            G.nodes[n]['RH'] *= 0.995

    # ── Edge Weight Assignment ───────────────────────────

    for u, v in G.edges():

      avg_lambda = (
          G.nodes[u]['lambda']
          + G.nodes[v]['lambda']
      ) / 2

      # Entanglement generation weight
      G[u][v]['weight'] = 1 / avg_lambda

      # Hardware compatibility reliability
      edge_rel = np.exp(
          -abs(
              G.nodes[u]['tau']
              - G.nodes[v]['tau']
          ) / 50
      )

      G[u][v]['edge_rel'] = edge_rel

      # Reliability-aware routing cost
      G[u][v]['cost'] = (
          G[u][v]['weight']
          / edge_rel
      )


    return G

In [ ]:
# ============================================================
# Quantum Network Graph Creation and Visualization
# ============================================================

from itertools import islice

# ── Line Graph Generation ────────────────────────────────

def create_line_graph(num_nodes,
                      bifunctional_prob=0.85):

    # Create simple path-based topology
    G = nx.path_graph(num_nodes)

    # Assign quantum-aware node parameters
    G = assign_node_parameters_quantum(
        G,
        bifunctional_prob
    )

    return G, list(range(num_nodes))


# ── General Graph Generation ─────────────────────────────

def create_general_graph(num_nodes,
                         extra_edges=4,
                         bifunctional_prob=0.85):

    """
    Controlled graph generation:
    - Long routing path
    - Limited shortcut edges
    - Stable QAOA optimization
    """

    # Start with line topology
    G = nx.path_graph(num_nodes)

    # Add limited local shortcut edges
    added = 0
    attempts = 0

    while added < extra_edges and attempts < 50:

        u = np.random.randint(0, num_nodes)
        v = np.random.randint(0, num_nodes)

        # Avoid excessive path shortening
        if (
            u != v
            and not G.has_edge(u, v)
            and abs(u - v) <= 3
        ):

            G.add_edge(u, v)

            added += 1

        attempts += 1

    # Assign quantum-aware parameters
    G = assign_node_parameters_quantum(
        G,
        bifunctional_prob
    )

   # ── Generate Top-k Candidate Paths ─────────────────

    k = 5

    paths = list(
        islice(
            nx.shortest_simple_paths(
                G,
                0,
                num_nodes - 1,
                weight='cost'
            ),
            k
        )
    )

    return G, paths

# ── Display Network Parameters ───────────────────────────

def display_network(G, path, title):

    print("=" * 80)
    print(title)
    print("=" * 80)

    print(f"{'Node':<5} {'Type':<5} {'HW':<6} "
          f"{'RT':>6} {'RH':>6} "
          f"{'PT':>6} {'PH':>6} "
          f"{'τ':>6} {'λ':>6} {'W':>6}")

    print("-" * 80)

    for node in path:

        d = G.nodes[node]

        print(f"{node:<5} "
              f"{d['type']:<5} "
              f"{d['hardware'][:4]:<6} "
              f"{d['RT']:>6.3f} "
              f"{d['RH']:>6.3f} "
              f"{d['PT']:>6.3f} "
              f"{d['PH']:>6.3f} "
              f"{d['tau']:>6.3f} "
              f"{d['lambda']:>6.3f} "
              f"{d['W']:>6.3f}")

    print("=" * 80)


# ── Graph Visualization ──────────────────────────────────

def plot_graph(G, path, ax, title):

    pos = nx.spring_layout(G, seed=42)

    # Node colors
    node_colors = [
        '#1D9E75'
        if G.nodes[n]['type'] == 'bi'
        else '#7A95B5'
        for n in G.nodes()
    ]

    # Highlight routing path
    path_edges = list(zip(path[:-1], path[1:]))

    edge_colors = [
        '#EF9F27'
        if e in path_edges or (e[1], e[0]) in path_edges
        else '#2E4060'
        for e in G.edges()
    ]

    edge_widths = [
        3
        if e in path_edges or (e[1], e[0]) in path_edges
        else 1
        for e in G.edges()
    ]

    nx.draw(
        G,
        pos,
        ax=ax,
        node_color=node_colors,
        edge_color=edge_colors,
        width=edge_widths,
        with_labels=True
    )

    ax.set_title(title)
    ax.axis('off')

# ── Fixed Seed for Reproducibility ───────────────────────

# np.random.seed(42)


# ── Create Quantum Networks ──────────────────────────────

G_line, path_line = create_line_graph(
    10,
    bifunctional_prob=0.85
)

G_gen, candidate_paths = create_general_graph(
    15,
    extra_edges=6,
    bifunctional_prob=0.85
)
path_gen = candidate_paths[0]

# ── Display Network Details ──────────────────────────────

print("\n>>> LINE GRAPH")

display_network(
    G_line,
    path_line,
    "Line Graph"
)

print("\n>>> GENERAL GRAPH")

display_network(
    G_gen,
    path_gen,
    "General Graph"
)


print("\n>>> TOP-k Candidate Paths")

for i, p in enumerate(candidate_paths):

    cost = nx.path_weight(
        G_gen,
        p,
        weight='cost'
    )

    print(
        f"Path {i+1}: {p} "
        f"| Cost = {cost:.4f}"
    )

# ── Plot Graphs ──────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_graph(
    G_line,
    path_line,
    axes[0],
    "Line Graph"
)

plot_graph(
    G_gen,
    path_gen,
    axes[1],
    "General Graph"
)

plt.show()

# Routing Algorithms

In [ ]:
# ============================================================
# MRAG — Memory & Rate Aware Greedy Algorithm
# ============================================================

import numpy as np


def mrag(G, path):

    """
    Memory & Rate Aware Greedy Routing

    Improvements over base greedy:
    - Considers quantum memory decay
    - Considers entanglement waiting time
    - Uses effective reliability (R_eff)
    """

    # End-to-end reliability
    Rse = 1.0

    # Total transmission delay
    cumulative_time = 0.0

    # Mode assignment along routing path
    mode_assignment = []

    # Store detailed routing statistics
    details = []

    # ── Traverse Routing Path ────────────────────────────

    for i, node in enumerate(path):

        d = G.nodes[node]

        RT = d['RT']
        RH = d['RH']

        PT = d['PT']
        PH = d['PH']

        tau = d['tau']
        lam = d['lambda']

        # Entanglement waiting time
        W = 1.0 / lam

        next_node = (
            path[i + 1]
            if i + 1 < len(path)
            else None
        )

        current_is_mono = (
            d['type'] == 'mono'
        )

        next_is_mono = (
            G.nodes[next_node]['type'] == 'mono'
            if next_node
            else False
        )

        # ── Teleportation Effective Reliability ─────────

        delta_T_T = PT + W

        decay_T = np.exp(
            -0.3*delta_T_T / tau
        )

        Reff_T = RT * decay_T

        # ── H2H Effective Reliability ───────────────────

        delta_T_H = PH

        decay_H = np.exp(
            -0.3*delta_T_H / tau
        )

        Reff_H = RH * decay_H

        # ── Routing Mode Selection ──────────────────────

        # Mono-functional nodes force teleportation
        if current_is_mono or next_is_mono:

            mode = 'T'

            R_used = Reff_T

            T_used = delta_T_T

        else:

            # Select mode with higher effective reliability
            if Reff_T >= Reff_H:

                mode = 'T'

                R_used = Reff_T

                T_used = delta_T_T

            else:

                mode = 'H2H'

                R_used = Reff_H

                T_used = delta_T_H

        # ── Update Global Metrics ───────────────────────

        cumulative_time += T_used

        Rse *= R_used

        mode_assignment.append(mode)

        # Store detailed node statistics
        details.append({

            'node': node,

            'type': d['type'],

            'mode': mode,

            'RT': RT,

            'RH': RH,

            'tau': tau,

            'lambda': lam,

            'W': W,

            'Reff_T': Reff_T,

            'Reff_H': Reff_H,

            'R_used': R_used,

            'T_used': T_used,

            'cum_time': cumulative_time,

            'Rse_run': Rse
        })

    return (
        Rse,
        cumulative_time,
        mode_assignment,
        details
    )

In [ ]:
# ============================================================
# Top-k MRAG Optimization
# ============================================================

def topk_mrag(G, candidate_paths):

    best_result = None

    best_Rse = -1

    all_results = []

    # ── Evaluate Each Candidate Path ─────────────────

    for idx, path in enumerate(candidate_paths):

        Rse, total_time, modes, details = mrag(
            G,
            path
        )

        cost = nx.path_weight(
            G,
            path,
            weight='cost'
        )

        result = {

            'path_id'    : idx + 1,

            'path'       : path,

            'Rse'        : Rse,

            'time'       : total_time,

            'modes'      : modes,

            'cost'       : cost,

            'path_length': len(path)
        }

        all_results.append(result)

        print("=" * 75)

        print(f"Candidate Path {idx+1}")

        print(f"Path         : {path}")

        print(f"Cost         : {cost:.4f}")

        print(f"Reliability  : {Rse:.6f}")

        print(f"Time         : {total_time:.4f}")

        print(f"Modes        : {modes}")

        print("=" * 75)


        # ── Select Best Reliability Path ─────────────

        if Rse > best_Rse:

            best_Rse = Rse

            best_result = result

    return best_result, all_results

In [ ]:
best_mrag, all_mrag = topk_mrag(
    G_gen,
    candidate_paths
)

print("\nBEST MRAG PATH")

print("-" * 60)

print("Path:", best_mrag['path'])

print(f"Reliability: {best_mrag['Rse']:.6f}")

print(f"Time: {best_mrag['time']:.4f}")

print("Modes:", best_mrag['modes'])

In [ ]:
# ============================================================
# MRADP — Memory & Rate Aware Dynamic Programming
# ============================================================

def mradp(G, path, T_constraint):

    """
    Memory & Rate Aware Dynamic Programming

    Features:
    - Considers memory decoherence
    - Considers entanglement waiting delay
    - Enforces transmission time constraint
    - Globally optimizes routing reliability
    """

    import numpy as np

    n = len(path)

    # Time discretization step
    time_step = 0.01

    T_steps = int(
        T_constraint / time_step
    )

    # DP table:
    # caching[i][t] = best solution
    caching = [
        [None] * (T_steps + 1)
        for _ in range(n)
    ]

    # ── Base Case ────────────────────────────────────────

    for t in range(T_steps + 1):

        caching[n - 1][t] = (
            1.0,
            []
        )

    # ── Bottom-Up DP Computation ────────────────────────

    for i in range(n - 2, -1, -1):

        node = path[i]

        next_node = path[i + 1]

        d = G.nodes[node]

        d_next = G.nodes[next_node]

        RT = d['RT']
        RH = d['RH']

        PT = d['PT']
        PH = d['PH']

        tau = d['tau']

        lam = d['lambda']

        # Entanglement waiting delay
        W = 1.0 / lam

        PT_steps = int(
            (PT + W) / time_step
        )

        PH_steps = int(
            PH / time_step
        )

        current_is_mono = (
            d['type'] == 'mono'
        )

        next_is_mono = (
            d_next['type'] == 'mono'
        )

        for t in range(T_steps + 1):

            best_Rse = 0.0

            best_modes = None

            # ── Try Teleportation ───────────────────────

            rem_T = t - PT_steps

            if (
                rem_T >= 0
                and caching[i + 1][rem_T]
                is not None
            ):

                R_next, modes_next = \
                    caching[i + 1][rem_T]

                delta_T = PT + W

                decay = np.exp(
                    -0.3 * delta_T / tau
                )

                R_T = RT * decay * R_next

                if R_T > best_Rse:

                    best_Rse = R_T

                    best_modes = (
                        ['T'] + modes_next
                    )

            # ── Try H2H Routing ─────────────────────────

            if (
                not current_is_mono
                and not next_is_mono
            ):

                rem_H = t - PH_steps

                if (
                    rem_H >= 0
                    and caching[i + 1][rem_H]
                    is not None
                ):

                    R_next, modes_next = \
                        caching[i + 1][rem_H]

                    delta_T = PH

                    decay = np.exp(
                        -0.3 * delta_T / tau
                    )

                    R_H = RH * decay * R_next

                    if R_H > best_Rse:

                        best_Rse = R_H

                        best_modes = (
                            ['H2H'] + modes_next
                        )

            if best_modes is not None:

                caching[i][t] = (
                    best_Rse,
                    best_modes
                )

    # ── Extract Optimal Solution ────────────────────────

    best_Rse = 0.0

    best_time = 0.0

    best_modes = []

    for t in range(T_steps + 1):

        if caching[0][t] is not None:

            R, modes = caching[0][t]

            if R > best_Rse:

                best_Rse = R

                best_time = (
                    t * time_step
                )

                best_modes = modes

    # Destination alignment
    best_modes = best_modes + ['T']

    return (
        best_Rse,
        best_time,
        best_modes
    )

In [ ]:
# ============================================================
# Top-k MRADP Optimization
# ============================================================

def topk_mradp(G,
               candidate_paths,
               T_constraint):

    best_result = None

    best_Rse = -1

    all_results = []

    # ── Evaluate Each Candidate Path ─────────────────

    for idx, path in enumerate(candidate_paths):

        Rse, total_time, modes = mradp(
            G,
            path,
            T_constraint
        )

        cost = nx.path_weight(
            G,
            path,
            weight='cost'
        )

        result = {

            'path_id'    : idx + 1,

            'path'       : path,

            'Rse'        : Rse,

            'time'       : total_time,

            'modes'      : modes,

            'cost'       : cost,

            'path_length': len(path)
        }

        all_results.append(result)

        print("=" * 75)

        print(f"Candidate Path {idx+1}")

        print(f"Path         : {path}")

        print(f"Cost         : {cost:.4f}")

        if Rse == 0:

          print("=" * 75)

          print(f"Candidate Path {idx+1}")

          print("No feasible solution under time constraint")

          print("=" * 75)

          continue

        print(f"Reliability  : {Rse:.6f}")

        print(f"Time         : {total_time:.4f}")

        print(f"Modes        : {modes}")

        print("=" * 75)

        # ── Select Best Global Path ───────────────

        if Rse > best_Rse:

            best_Rse = Rse

            best_result = result

    return best_result, all_results

In [ ]:
best_mradp, all_mradp = topk_mradp(
    G_gen,
    candidate_paths,
    T_constraint=5.0
)

print("\nBEST MRADP PATH")

print("-" * 60)

print("Path:", best_mradp['path'])

print(f"Cost: {best_mradp['cost']:.4f}")

print(f"Reliability: {best_mradp['Rse']:.6f}")

print(f"Time: {best_mradp['time']:.4f}")

print("Modes:", best_mradp['modes'])

# QAOA-based Quantum Routing Optimization

In [ ]:
# ============================================================
# QAOA Circuit Construction
# ============================================================

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.visualization import circuit_drawer


def create_qaoa_circuit_p1(n_qubits=4):

    """
    Create a single-layer (p=1) QAOA circuit

    Components:
    - Superposition initialization
    - Cost Hamiltonian
    - Mixer Hamiltonian
    """

    qc = QuantumCircuit(n_qubits)

    # QAOA trainable parameters
    gamma = Parameter('γ')

    beta = Parameter('β')

    # ========================================================
    # Step 1: Superposition Initialization
    # ========================================================

    # Create equal superposition of all routing states
    qc.h(range(n_qubits))

    # ========================================================
    # Step 2: Cost Hamiltonian
    # ========================================================

    # Single-qubit cost terms
    for i in range(n_qubits):

        qc.rz(2 * gamma, i)

    # Interaction cost terms
    for i in range(n_qubits - 1):

        qc.cx(i, i + 1)

        qc.rz(2 * gamma, i + 1)

        qc.cx(i, i + 1)

    # ========================================================
    # Step 3: Mixer Hamiltonian
    # ========================================================

    # Explore alternative routing configurations
    for i in range(n_qubits):

        qc.rx(2 * beta, i)

    return qc


# ============================================================
# Create and Display QAOA Circuit
# ============================================================

qc = create_qaoa_circuit_p1(4)

qc.draw('mpl')

In [ ]:
# ============================================================
# Time-Aware QAOA with Classical Parameter Optimization
# ============================================================

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit_aer import AerSimulator

import numpy as np

from scipy.optimize import minimize


# ============================================================
# Identify Bi-functional Decision Nodes
# ============================================================

def get_bi_nodes(G, path):

    bi_nodes = []

    bi_indices = []

    for i, node in enumerate(path):

        if G.nodes[node]['type'] == 'bi':

            next_node = (
                path[i + 1]
                if i + 1 < len(path)
                else None
            )

            if next_node is not None:

                bi_nodes.append(node)

                bi_indices.append(i)

    return bi_nodes, bi_indices


# ============================================================
# Evaluate Routing Cost for a Bitstring
# ============================================================

def evaluate_bitstring(G,
                       path,
                       bitstring,
                       bi_indices):

    Rse = 1.0

    cumulative_time = 0.0

    idx = 0

    for i, node in enumerate(path):

        d = G.nodes[node]

        tau = d['tau']

        lam = d['lambda']

        W = 1.0 / lam

        next_node = (
            path[i + 1]
            if i + 1 < len(path)
            else None
        )

        current_is_mono = (
            d['type'] == 'mono'
        )

        next_is_mono = (
            G.nodes[next_node]['type'] == 'mono'
            if next_node
            else False
        )

        # Forced teleportation
        if (
            current_is_mono
            or next_is_mono
            or i not in bi_indices
        ):

            mode = 'T'

        else:

            mode = (
                'H2H'
                if bitstring[idx] == 1
                else 'T'
            )

            idx += 1

        # Memory decoherence
        decay = np.exp(
            -0.3 * cumulative_time / tau
        )

        # Reliability selection
        if mode == 'T':

            R_used = d['RT'] * decay

            T_used = d['PT'] + W

        else:

            R_used = d['RH'] * decay

            T_used = d['PH']

        Rse *= R_used

        cumulative_time += T_used

    # Minimize routing cost
    return -np.log(Rse + 1e-12)


# ============================================================
# Compute Time-Aware QAOA Cost Coefficients
# ============================================================

def compute_time_aware_coeffs(G,
                              path,
                              bi_indices):

    c_T_all = []

    c_H_all = []

    cumulative_time = 0.0

    for i, node in enumerate(path):

        d = G.nodes[node]

        tau = d['tau']

        lam = d['lambda']

        W = 1.0 / lam

        # Decoherence factor
        decay = np.exp(
            -0.3 * cumulative_time / tau
        )

        c_T_all.append(
            -np.log(d['RT'] * decay)
        )

        c_H_all.append(
            -np.log(d['RH'] * decay)
        )

        cumulative_time += (
            d['PT'] + W
        )

    c_T = [
        c_T_all[i]
        for i in bi_indices
    ]

    c_H = [
        c_H_all[i]
        for i in bi_indices
    ]

    n = len(c_T)

    # Interaction matrix
    J = np.zeros((n, n))

    for i in range(n):

        for j in range(i + 1, n):

            J[i][j] = abs(
                c_H[j] - c_T[j]
            )

    return (
        np.array(c_T),
        np.array(c_H),
        J
    )


# ============================================================
# Build Time-Aware QAOA Circuit
# ============================================================

def build_time_aware_qaoa(n,
                          c_T,
                          c_H,
                          J):

    qc = QuantumCircuit(n, n)

    # Trainable QAOA parameters
    gamma = Parameter('γ')

    beta = Parameter('β')

    # ── Initialization ───────────────────────────────────

    # Create superposition of routing states
    qc.h(range(n))

    # ── Cost Hamiltonian ────────────────────────────────

    # Local cost terms
    for i in range(n):

        cost_diff = c_H[i] - c_T[i]

        qc.rz(
            2 * gamma * cost_diff,
            i
        )

    # Interaction terms
    for i in range(n):

        for j in range(i + 1, n):

            if J[i][j] != 0:

                qc.cx(i, j)

                qc.rz(
                    2 * gamma * J[i][j],
                    j
                )

                qc.cx(i, j)

    # ── Mixer Hamiltonian ───────────────────────────────

    for i in range(n):

        qc.rx(
            2 * beta,
            i
        )

    # Measurements
    qc.measure(range(n), range(n))

    return qc


# ============================================================
# QAOA Objective Function
# ============================================================

def qaoa_objective(params,
                   G,
                   path,
                   bi_indices):

    gamma, beta = params

    c_T, c_H, J = compute_time_aware_coeffs(
        G,
        path,
        bi_indices
    )

    n = len(c_T)

    qc = build_time_aware_qaoa(
        n,
        c_T,
        c_H,
        J
    )

    qc = qc.assign_parameters({

        'γ': gamma,

        'β': beta
    })

    # Quantum simulation
    sim = AerSimulator()

    result = sim.run(
        qc,
        shots=512
    ).result()

    counts = result.get_counts()

    cost = 0

    # Expected routing cost
    for bitstring, count in counts.items():

        bits = [int(b) for b in bitstring]

        c = evaluate_bitstring(
            G,
            path,
            bits,
            bi_indices
        )

        cost += c * count

    return cost / 512


# ============================================================
# Classical Optimization of QAOA Parameters
# ============================================================

def optimize_qaoa(G, path):

    bi_nodes, bi_indices = \
        get_bi_nodes(G, path)

    print("=" * 60)

    print("OPTIMIZING QAOA PARAMETERS")

    print("=" * 60)

    # COBYLA classical optimizer
    res = minimize(

        qaoa_objective,

        x0=[0.5, 0.5],

        args=(G, path, bi_indices),

        method='COBYLA'
    )

    print("Optimized γ, β:", res.x)

    return res.x


# ============================================================
# Final QAOA Execution
# ============================================================

def run_qaoa_single_path(G, path):

    bi_nodes, bi_indices = \
        get_bi_nodes(G, path)

    n = len(bi_indices)

    # Optimize parameters
    gamma, beta = optimize_qaoa(
        G,
        path
    )

    # Compute QAOA coefficients
    c_T, c_H, J = \
        compute_time_aware_coeffs(
            G,
            path,
            bi_indices
        )

    # Build final circuit
    qc = build_time_aware_qaoa(
        n,
        c_T,
        c_H,
        J
    )

    qc = qc.assign_parameters({

        'γ': gamma,

        'β': beta
    })

    # print("\nFINAL QAOA CIRCUIT:\n")

    # display(qc.draw('mpl'))

    # Final execution
    sim = AerSimulator()

    result = sim.run(
        qc,
        shots=1024
    ).result()

    counts = result.get_counts()

    # Best routing configuration
    best_state = max(
        counts,
        key=counts.get
    )

    print("\nFinal Counts:", counts)

    print("Best bitstring:", best_state)

    return best_state, counts


# ============================================================
# Run Final QAOA Optimization
# ============================================================

best_state, counts = run_qaoa_single_path(
    G_gen,
    path_gen
)

In [ ]:
# ============================================================
# Top-k QAOA Optimization
# ============================================================

def topk_qaoa(G,
              candidate_paths):

    best_result = None

    best_Rse = -1

    all_results = []

    for idx, path in enumerate(candidate_paths):

        print("\n" + "=" * 80)

        print(f"QAOA Optimization — Candidate Path {idx+1}")

        print("=" * 80)

        best_state, counts = run_qaoa_single_path(
            G,
            path
        )

        bits = [int(b) for b in best_state[::-1]]

        routing_cost = evaluate_bitstring(
            G,
            path,
            bits,
            get_bi_nodes(G, path)[1]
        )

        Rse = np.exp(-routing_cost)

        result = {

            'path_id' : idx + 1,

            'path'    : path,

            'state'   : best_state,

            'counts'  : counts,

            'prob'    : Rse
        }

        all_results.append(result)

        print(f"Dominant State : {best_state}")

        print(f"Routing Reliability : {Rse:.6f}")

        # Select globally best path
        if Rse > best_Rse:

            best_Rse = Rse

            best_result = result

    return best_result, all_results

In [ ]:
best_qaoa, all_qaoa = topk_qaoa(
    G_gen,
    candidate_paths
)

print("\nBEST QAOA PATH")

print("-" * 60)

print("Path:", best_qaoa['path'])

print("Best State:",
      best_qaoa['state'])

print("Routing Reliability:",
      f"{best_qaoa['prob']:.4f}")